# Temporal Alignment — News and Liquidity Data using OHLCV

This notebook aligns the cleaned GDELT news dataset with the hourly WTI price and 
liquidity data from Yahoo Finance. Each news article is matched to the closest 
trading hour using timestamp rounding. Articles published outside trading hours 
(nights, weekends) are discarded as no liquidity data is available for those 
periods. The resulting dataset merges article metadata with contemporaneous 
liquidity metrics (log volume, price range, Amihud ratio), and constitutes the 
central modeling dataset for the thesis. It is saved to 
`01_data/processed/gdelt_wti_aligned.csv`.

### General imports

In [21]:
import pandas as pd
import numpy as np

### Load hourly price data

In [22]:
df_price = pd.read_csv("../01_data/raw/price/wti_hourly_raw.csv", 
                       index_col='Datetime', 
                       parse_dates=True)
df_price.index = pd.to_datetime(df_price.index, utc=True)

print(f"Price records: {len(df_price)}")
print(f"Range: {df_price.index.min()} to {df_price.index.max()}")

Price records: 11219
Range: 2024-03-11 10:00:00+00:00 to 2026-03-11 10:00:00+00:00


### Load news and assign to next available trading hour

Loads the cleaned GDELT news dataset and assigns each article to its next 
available trading hour in the WTI price series.

**Why ceil to the next trading hour instead of rounding to the nearest:**
Rounding to the nearest hour can assign an article to a trading hour that 
occurred *before* the article was published — for example, an article published 
at 14:37 would be matched to the 14:00 candle, meaning the market data would 
precede the news. This violates causality. By always moving forward to the next 
available trading hour, we ensure the market data always comes *after* the article.

**Why forward-assign instead of discarding off-hours articles:**
Previous versions of this pipeline discarded articles published outside NYMEX 
trading hours (nights, weekends, daily break). However, this introduces an 
information loss — a bearish article published Saturday afternoon is still read 
by traders over the weekend and acted upon the moment the market reopens Sunday 
night. Discarding it would ignore this information accumulation effect.

The updated approach assigns off-hours articles to the next available trading 
hour instead:
- Article published Saturday 2:00 PM UTC → assigned to Sunday 11:00 PM UTC (market open)
- Article published during daily break (10:30 PM UTC) → assigned to 11:00 PM UTC
- Article published at 3:00 AM weekday → assigned to 4:00 AM (next trading hour)

Only articles published after the end of the price dataset are discarded, 
as no future trading hour exists to assign them to.

In [23]:
# Load news
df_news = pd.read_csv("../01_data/processed/gdelt_wti_with_body.csv", parse_dates=['datetime'])
df_news['datetime'] = pd.to_datetime(df_news['datetime'], utc=True)

# Get sorted price index for lookup
price_index = df_price.index.sort_values()

# For each article, find the next available trading hour
def get_next_trading_hour(timestamp, price_index):
    future_hours = price_index[price_index >= timestamp]
    if len(future_hours) > 0:
        return future_hours[0]
    return None

print("Assigning articles to next available trading hour...")
df_news['datetime_hour'] = df_news['datetime'].apply(
    lambda ts: get_next_trading_hour(ts, price_index)
)

# Discard only articles that fall after the end of the price dataset
before = len(df_news)
df_news = df_news.dropna(subset=['datetime_hour']).reset_index(drop=True)
after = len(df_news)

print(f"Articles before: {before}")
print(f"Discarded (beyond price range): {before - after}")
print(f"Articles after: {after}")

Assigning articles to next available trading hour...
Articles before: 16326
Discarded (beyond price range): 0
Articles after: 16326


### Merge news articles with hourly price and liquidity data

Each news article has been assigned to its next available trading hour via 
`datetime_hour`. This cell performs a left join between the news dataset and 
the hourly WTI price series on that timestamp, attaching the following market 
variables to each article:

- **Close:** WTI closing price at the matched trading hour
- **Volume:** Number of contracts traded at the matched trading hour
- **log_volume:** Natural log of Volume — primary liquidity variable
- **price_range:** High − Low — intraday volatility proxy (Parkinson, 1980)
- **log_return:** log(Close_t / Close_{t-1}) — hourly log return
- **amihud:** |log_return| / Volume — illiquidity ratio (Amihud, 2002)

Articles where no matching price record exists (e.g. articles assigned beyond 
the end of the price dataset) will have NaN values for all market variables 
and are discarded in the following cell.

In [24]:
# Merge news with price data on datetime_hour
df_price_reset = df_price.copy()
df_price_reset.index.name = 'datetime_hour'
df_price_reset = df_price_reset.reset_index()

# Compute liquidity features
df_price_reset['log_volume'] = np.log(df_price_reset['Volume'] + 1)
df_price_reset['price_range'] = df_price_reset['High'] - df_price_reset['Low']
df_price_reset['log_return'] = np.log(df_price_reset['Close'] / df_price_reset['Close'].shift(1))
df_price_reset['amihud'] = df_price_reset['log_return'].abs() / (df_price_reset['Volume'] + 1)

df_news = df_news.merge(
    df_price_reset[['datetime_hour', 'Close', 'Volume', 'log_volume', 'price_range', 'log_return', 'amihud']],
    on='datetime_hour',
    how='left'
)

print(f"Articles after merge: {len(df_news)}")
print(f"Without price data: {df_news['Volume'].isna().sum()}")

Articles after merge: 16326
Without price data: 0


### Discard articles without price data

Articles published outside NYMEX trading hours (nights and weekends) cannot be matched to a trading hour and are therefore discarded. Since no market activity is observable at those times, including them would not contribute meaningful liquidity measurements to the analysis.

In [25]:
df_final = df_news.dropna(subset=['Close', 'Volume']).reset_index(drop=True)

# Discard articles published before price data starts and zero volume hours
price_start = df_price.index.min()
df_final = df_final[df_final['datetime'] >= price_start].reset_index(drop=True)
df_final = df_final[df_final['Volume'] > 0].reset_index(drop=True)

print(f"Articles after fixing pre-price and zero volume: {len(df_final)}")
print(f"Temporal range: {df_final['datetime'].min()} to {df_final['datetime'].max()}")
print(f"\nLiquidity statistics:")
print(df_final[['log_volume', 'price_range', 'amihud']].describe().round(4))

Articles after fixing pre-price and zero volume: 13691
Temporal range: 2024-03-11 10:30:00+00:00 to 2026-03-01 00:30:00+00:00

Liquidity statistics:
       log_volume  price_range      amihud
count  13691.0000   13691.0000  13691.0000
mean       8.4840       0.3893      0.0000
std        1.5472       0.3174      0.0000
min        0.6931       0.0000      0.0000
25%        7.4894       0.2000      0.0000
50%        8.6444       0.3200      0.0000
75%        9.6444       0.5000      0.0000
max       12.1323       5.8700      0.0021


In [27]:
# Keep only articles published close to their assigned trading hour
df_final['assignment_gap'] = (
    df_final['datetime_hour'] - df_final['datetime']
).dt.total_seconds() / 3600

df_contemporaneous = df_final[df_final['assignment_gap'] < 2].copy()
print(f"Contemporaneous articles: {len(df_contemporaneous)}")
print(f"Off-hours articles removed: {len(df_final) - len(df_contemporaneous)}")

# Check the distribution of assignment gaps
print("Assignment gap distribution:")
print(df_final['assignment_gap'].describe().round(2))

print("\nGap buckets:")
print(f"< 1 hour:    {(df_final['assignment_gap'] < 1).sum()}")
print(f"1-2 hours:   {((df_final['assignment_gap'] >= 1) & (df_final['assignment_gap'] < 2)).sum()}")
print(f"2-6 hours:   {((df_final['assignment_gap'] >= 2) & (df_final['assignment_gap'] < 6)).sum()}")
print(f"6-12 hours:  {((df_final['assignment_gap'] >= 6) & (df_final['assignment_gap'] < 12)).sum()}")
print(f"12-24 hours: {((df_final['assignment_gap'] >= 12) & (df_final['assignment_gap'] < 24)).sum()}")
print(f"> 24 hours:  {(df_final['assignment_gap'] >= 24).sum()}")

Contemporaneous articles: 11317
Off-hours articles removed: 2374
Assignment gap distribution:
count    13691.00
mean         4.74
std         11.72
min          0.00
25%          0.25
50%          0.50
75%          0.75
max         73.50
Name: assignment_gap, dtype: float64

Gap buckets:
< 1 hour:    11174
1-2 hours:   143
2-6 hours:   302
6-12 hours:  409
12-24 hours: 469
> 24 hours:  1194


### Save data

In [28]:
# Save full aligned dataset (all articles)
df_final.to_csv("../01_data/processed/gdelt_wti_aligned.csv", index=False)
print(f"Full dataset saved: {len(df_final)} articles")

# Save contemporaneous only (gap <= 2 hours)
df_contemporaneous = df_final[df_final['assignment_gap'] <= 2].copy().reset_index(drop=True)
df_contemporaneous.to_csv("../01_data/processed/gdelt_wti_aligned_contemporaneous.csv", index=False)
print(f"Contemporaneous dataset saved: {len(df_contemporaneous)} articles")

Full dataset saved: 13691 articles
Contemporaneous dataset saved: 11332 articles
